<a href="https://www.kaggle.com/code/leonardoterra/behavior-traits-eda?scriptVersionId=250003234" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

 **Dataset**
--
    Data Science EDA
    
### **Data Description**
This dataset compiles information on aviation accidents and associated fatalities worldwide from 1970 through 2025. It includes annual counts of flight crashes and total death tolls, offering insights into trends in aviation safety over five decades.

### **Features**
    Year: Calendar year from 1970 to 2025.
    Crashes: Total number of recorded aviation accidents in that year.
    Deaths: Total number of fatalities resulting from aviation accidents in that year.

## **Import Libraries**

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

import warnings 
warnings.filterwarnings('ignore')

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter

# **EDA**

## **Data Exploration**
    During this step we can explore the shape of the data we are working with. It's usual to check and deal with the number of columns and rows, missing values, outliers and other inconsistencies that might appear.
--

In [ ]:
df = pd.read_csv('/kaggle/input/extrovert-vs-introvert-behavior-data/personality_dataset.csv')
df

In [ ]:
df.dtypes

In [ ]:
df.isna().mean() # Evaluating the % of missing values in every column

In [ ]:
# Getting numeric columns to assess outliers
numeric_df_columns = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
numeric_df_columns

##### **BoxPlots to assess outliers and missing values**

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(16, 4))

ax[0,0] = sns.boxplot(data=df, x= 'Time_spent_Alone', ax=ax[0,0])
ax[0,1] = sns.boxplot(data=df, x= 'Social_event_attendance', ax=ax[0,1])
ax[0,2] = sns.boxplot(data=df, x= 'Going_outside', ax=ax[0,2])
ax[1,0] = sns.boxplot(data=df, x= 'Friends_circle_size', ax=ax[1,0])
ax[1,1] = sns.boxplot(data=df, x= 'Post_frequency', ax=ax[1,1])

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

In [ ]:
#Getting Object columns to assess missing values

object_df_columns = df.select_dtypes(include='object').columns.tolist()
object_df_columns.remove('Personality')
df[object_df_columns].mode()

In [ ]:
# Since we have no outliers and <5% are missing, we will input the mode in the Object columns and the mean in the numeric columns. 

df_v2 = df
df_v2[object_df_columns] = df_v2[object_df_columns].apply(lambda col: col.fillna(col.mode()[0]))
df_v2[numeric_df_columns] = df_v2[numeric_df_columns].apply(lambda col: col.fillna(col.mean()))
df_v2.isna().mean() # checking the number of outliers after treatment

##### **Encoding Categorical features**

In [ ]:
df_v2['Stage_fear'] = df_v2['Stage_fear'].map({'Yes':1, 'No':0})
df_v2['Drained_after_socializing'] = df_v2['Drained_after_socializing'].map({'Yes':1, 'No':0})
df_v2['Personality'] = df_v2['Personality'].map({'Extrovert':1, 'Introvert':0})
df_v2.head()

## **Statistical Analysis**
    This step seeks to understand the interaction and correlation among the features. Here it's possible to check multicolinearity, plot visulizations and highlight key points.
--

##### **Visualization plots to describe the data**

In [ ]:
fig, ax = plt.subplots(nrows=2, ncols=3, figsize=(12, 7))

ax[0,0] = sns.boxplot(data=df_v2, x='Personality', y='Time_spent_Alone', ax=ax[0,0], )
ax[0,0].set_xticklabels(['Introvert', 'Extrovert'])

ax[0,1] = sns.boxplot(data=df_v2, x='Personality', y= 'Social_event_attendance', ax=ax[0,1])
ax[0,1].set_xticklabels(['Introvert', 'Extrovert'])

ax[0,2] = sns.boxplot(data=df_v2, x='Personality', y= 'Going_outside', ax=ax[0,2])
ax[0,2].set_xticklabels(['Introvert', 'Extrovert'])

ax[1,0] = sns.countplot(data=df_v2, y= 'Drained_after_socializing', hue='Personality', ax=ax[1,0])
legend = ax[1,0].legend(labels=['Extrovert','Introvert'], fontsize='9')
legend.set_frame_on(False)

ax[1,1] = sns.boxplot(data=df_v2, x='Personality', y= 'Friends_circle_size', ax=ax[1,1])
ax[1,1].set_xticklabels(['Introvert', 'Extrovert'])

ax[1,2] = sns.boxplot(data=df_v2, x='Personality', y= 'Post_frequency', ax=ax[1,2])
ax[1,2].set_xticklabels(['Introvert', 'Extrovert'])

sns.despine(left=False, right=True, top=True, bottom=False)
plt.tight_layout()

##### **Correlation of features**

In [ ]:
corr = df_v2.corr()
mask = np.triu(np.ones_like(corr, dtype=bool)) # Mask to remove the upper side and facilitate visualization

plt.figure(figsize=(10,6))
sns.heatmap(corr, mask=mask, annot=True, cmap='coolwarm', center=0)

sns.despine(left=False, right=True, top=True, bottom=False)
plt.title("Correlation Analysis", fontsize=9)
plt.tight_layout()

##### **Descriptive statistics of the Personality groups**

In [ ]:
introverts_statistics = df_v2['Personality'] == 0
introverts = df_v2[introverts_statistics].describe().round(2).reset_index().drop('Personality', axis=1)
introverts

In [ ]:
extroverts_statistics = df_v2['Personality'] == 1
extroverts = df_v2[extroverts_statistics].describe().round(2).reset_index().drop('Personality', axis=1)
extroverts

## **Insights**
--

### **Statistics**:
* Extrovert people tend to spend on average 3 times more time speaking with other people, posting and going outside than introverts. Introverts on the other hand tend to stay alone and don't have big friendship circles. This results in stage fear, less social event attendance and also the out of energy feeling when they need to interact with more people.
* The dependent Y (Personality) variable fits too well with the data collected. It means that it's easy to understand the correlation of the X features with the Y, and based on this data, a ML model would perform very well, however would also present overfitting tendencies.

# **Thank You for taking the time to view this Notebook**!
​
If you found this analysis useful and have any feedback or suggestions, don't hesitate to contact me! We are here to learn!